# 03 SFT：从可验证轨迹到规则记忆与执行

本节研究监督数据和推理轨迹如何让 `Qwen/Qwen3-4B-Thinking-2507` 记忆、检索并执行课程提供的规则。核心对照是 Direct Full-Trace、Joint Drill + Full-Trace 和 Two-stage Memorization → Execution；LoRA / QLoRA 是同一 Notebook 内的参数高效分支。

当前 DGX Spark 阶段只进行数据审计、Label Masking、Loss 和小模型 CPU Smoke Test。模型加载、训练和显存测量必须在明天的 RTX 5090 官方 Profile 中执行；本 Notebook 默认 `RUN_SFT=False`。

## 学习目标

- 判断教师轨迹是否能从 Prompt 和公开先验推导；
- 将 Prompt Token Mask 为 `-100`，只对 Assistant Completion 计算 Cross Entropy；
- 比较 Direct、Joint 和 Two-stage 的训练数据与 Token Budget；
- 理解 BF16 LoRA 与 NF4 QLoRA 的工程边界；
- 使用自定义 PyTorch 训练循环保存可交接的 Adapter 和日志。

## 1. 实验控制与数据边界

真实数据划分沿用第一个 Notebook。当前默认只使用已经独立筛查的 `synthetic_accepted.jsonl` 构造参考轨迹；它们来自 Train 侧先验，未读取 Validation / Holdout 答案。若未来加入真实 Train 轨迹，必须经过同样的 Teacher Solvability 和 Trace Fidelity 审计。

| 策略 | 训练内容 | 研究问题 |
|---|---|---|
| SFT-A Direct | 完整可验证轨迹 | 直接学习规则执行 |
| SFT-B Joint | Recall Drill + 完整轨迹混合 | 记忆与执行是否互补 |
| SFT-C Two-stage | 先 Drill，再 Full Trace | Memorization → Execution 是否有序收益 |
| SFT-D Targeted Drill | 从错误分析选择局部 Drill | 选做，不无差别重复 |

比较时记录训练 Token Budget、更新步数和数据来源；两阶段多跑的 Token 必须单独报告，不能把训练量增加误认为顺序收益。

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import time
from dataclasses import dataclass
from collections import Counter
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
SYNTHETIC_ACCEPTED_PATH = PROJECT_ROOT / "data" / "processed" / "synthetic" / "synthetic_accepted.jsonl"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "sft"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "processed" / "checkpoints"
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
BASE_MODEL_PATH = os.environ.get("QWEN3_MODEL_PATH", MODEL_ID)
TOKENIZER_PATH = os.environ.get("QWEN3_TOKENIZER_PATH", BASE_MODEL_PATH)
SEED = 42
RUN_SFT = os.environ.get("RUN_SFT", "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")       # smoke / full
MAX_LENGTH = int(os.environ.get("SFT_MAX_LENGTH", "1024"))
MICRO_BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
MAX_TRAIN_RECORDS = 24 if RUN_MODE == "smoke" else 512
SMOKE_QUANTIZATION_MODE = "nf4_qlora"
TRAINING_PROFILE = "official-q-qlora-or-pt-sft-cuda"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not TRAIN_PATH.exists():
    raise FileNotFoundError("缺少第一个 Notebook 生成的 Train 划分")
train_df = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)

print({
    "model_id": MODEL_ID,
    "device_for_cpu_smoke": DEVICE,
    "run_sft": RUN_SFT,
    "run_mode": RUN_MODE,
    "max_length": MAX_LENGTH,
    "train_rows": len(train_df),
})

## 2. Prompt-derivable Teacher Trace

Solver 计算出正确答案，并不自动意味着轨迹适合学习。每条轨迹要同时满足：最终答案正确、每一步引用当前 Prompt 或公开先验、变量先定义后使用、没有把 `hidden_rule` 写进 Completion、没有跳过全部示例验证、长度在预算内。

下面的 Trace Builder 只读取合成记录中的公开 Prompt、Solver Trace 和最终答案。`hidden_rule` 只留在元数据审计中，绝不进入用户 Prompt 或 Assistant Completion。

In [ ]:
def load_screened_synthetic_records(path: Path = SYNTHETIC_ACCEPTED_PATH) -> list[dict[str, object]]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _format_trace(sample: dict[str, object]) -> str:
    task = str(sample.get("task_type", ""))
    trace = sample.get("trace", {}) if isinstance(sample.get("trace", {}), dict) else {}
    answer = str(sample.get("answer", "")).strip()
    if task == "gravity":
        g = trace.get("g_ols", "inferred from examples")
        query_t = trace.get("query_t", "the Query time")
        query_raw = trace.get("query_raw", "computed from the inferred g")
        reasoning = (
            "1. Parse all public time-distance examples.\n"
            "2. Fit g = 2*sum(t^2*d)/sum(t^4) using only those examples.\n"
            f"3. The fitted g is {g}; apply distance = 0.5*g*t^2 to Query t={query_t}.\n"
            f"4. The unrounded Query distance is {query_raw}; format the final numeric answer."
        )
    elif task == "equation_numeric":
        candidate_rules = trace.get("candidate_rules_by_operator", {})
        outputs = trace.get("query_outputs", [])
        reasoning = (
            "1. Parse every AB<operator>CD example.\n"
            f"2. Keep only public 24-rule candidates consistent with each example: {candidate_rules}.\n"
            f"3. Apply the normal/flip family and semantic-group constraints; valid assignment count is {trace.get('valid_assignment_count', 'unknown')}.\n"
            f"4. Apply the selected candidates to the Query; valid Query outputs are {outputs}.\n"
            "5. Keep the answer only when the verifier finds one unique Query output."
        )
    elif task == "bit_manipulation":
        ident = trace.get("identifiability", {})
        expression = trace.get("expression", "a candidate expression inferred from examples")
        reasoning = (
            "1. Parse all 8-bit input-output examples.\n"
            "2. Enumerate legal DSL expressions and discard candidates that fail any example.\n"
            f"3. The selected semantic candidate is {expression}; identifiability summary is {ident}.\n"
            "4. Apply that candidate to the Query and format exactly eight bits."
        )
    else:
        reasoning = "1. Use the public task specification.\n2. Verify every example before computing the Query."
    return reasoning + f"\nFinal answer: \\boxed{{{answer}}}"


def make_trace_record(sample: dict[str, object]) -> dict[str, object]:
    completion = _format_trace(sample)
    return {
        "prompt": str(sample.get("prompt", "")),
        "reasoning": completion.rsplit("\nFinal answer:", 1)[0],
        "final_answer": str(sample.get("answer", "")).strip(),
        "completion": completion,
        "task_type": str(sample.get("task_type", "")),
        "trace_type": "full_trace",
        "source": "screened_synthetic_train_prior",
        "source_id": str(sample.get("candidate_id", "")),
        "verification_status": str(sample.get("verification_status", "")),
        "hidden_rule_used_in_completion": False,
        "token_length_proxy": len(completion.split()),
    }


def audit_trace_record(record: dict[str, object], max_tokens: int = MAX_LENGTH) -> dict[str, object]:
    completion = str(record.get("completion", ""))
    reasoning = str(record.get("reasoning", ""))
    answer = str(record.get("final_answer", "")).strip()
    checks = {
        "schema_ok": all(record.get(field) is not None for field in ("prompt", "completion", "task_type", "source_id")),
        "final_answer_present": bool(answer) and f"{answer}}}" in completion,
        "trace_nonempty": bool(reasoning.strip()),
        "prompt_derivable": "hidden_rule" not in completion.lower() and "ground truth" not in completion.lower(),
        "verification_status_ok": record.get("verification_status") == "accepted",
        "length_ok": int(record.get("token_length_proxy", len(completion.split()))) <= max_tokens,
        "task_supported": record.get("task_type") in {"gravity", "equation_numeric", "bit_manipulation"},
    }
    return {"accepted": all(checks.values()), "checks": checks, "rejection_reasons": [name for name, ok in checks.items() if not ok]}


def build_full_trace_records(limit: int | None = None) -> list[dict[str, object]]:
    samples = load_screened_synthetic_records()
    if limit is not None and samples:
        # Stratify from the complete accepted artifact before applying the limit;
        # slicing first would silently produce Gravity-only Smoke data.
        task_order = ("gravity", "equation_numeric", "bit_manipulation")
        grouped = {task: [sample for sample in samples if sample.get("task_type") == task] for task in task_order}
        target = int(limit)
        base = target // len(task_order)
        remainder = target % len(task_order)
        selected = []
        for index, task in enumerate(task_order):
            take = base + (1 if index < remainder else 0)
            selected.extend(grouped[task][:take])
        samples = selected
    elif limit is not None:
        samples = samples[:limit]
    records = [make_trace_record(sample) for sample in samples]
    audited = []
    for record in records:
        audit = audit_trace_record(record)
        record["trace_audit"] = audit
        if audit["accepted"]:
            audited.append(record)
    return audited

FULL_TRACE_RECORDS = build_full_trace_records(MAX_TRAIN_RECORDS)
print({
    "screened_synthetic_available": len(load_screened_synthetic_records()),
    "full_trace_records_after_audit": len(FULL_TRACE_RECORDS),
    "by_task": pd.Series([row["task_type"] for row in FULL_TRACE_RECORDS]).value_counts().to_dict() if FULL_TRACE_RECORDS else {},
})

## 3. Recall Drill、Joint 和 Two-stage 数据

Recall Drill 只训练公开规则定义、解析模式和格式约定，不包含某道题的隐藏规则。Joint 将 Drill 与完整轨迹混合；Two-stage 先只优化 Drill，再只优化完整轨迹。下面的计数和 Token 代理用于公平记录，不代表实际 GPU 训练结果。

In [ ]:
EQUATION_DRILL_ITEMS = [
    ("equation_parse", "AB_CD parses AB and CD directly; BA_DC reverses both two-digit operands."),
    ("equation_global_constraint", "Non-join rules share one normal or flip family and use at most one rule per semantic group."),
    ("equation_guess", "A missing Query operator is Guess; retain it only when all valid assignments agree on the Query output."),
    ("equation_join", "join concatenates AB then CD; join_swap concatenates CD then AB."),
]
BIT_DRILL_ITEMS = [
    ("bit_mask", "Every Bit Manipulation result is masked to an unsigned 8-bit value."),
    ("bit_rotate", "ROTL and ROTR rotate within eight bits; a shift amount is in 0..7."),
    ("bit_boolean", "AND, OR, XOR and NOT operate on the 8-bit expression value."),
]
GRAVITY_DRILL_ITEMS = [
    ("gravity_formula", "distance = 0.5*g*t^2, with t in seconds, distance in meters, and g inferred per question."),
    ("gravity_ols", "g can be estimated from public examples as 2*sum(t^2*d)/sum(t^4)."),
]


def build_recall_drill_records() -> list[dict[str, object]]:
    items = []
    for task, pairs in (("equation_numeric", EQUATION_DRILL_ITEMS), ("bit_manipulation", BIT_DRILL_ITEMS), ("gravity", GRAVITY_DRILL_ITEMS)):
        for name, statement in pairs:
            items.append({
                "prompt": f"Recall Drill ({task}): state the public rule for {name}.",
                "reasoning": "Recall only the course-provided prior; do not infer a sample-specific hidden rule.",
                "final_answer": statement,
                "completion": f"Rule: {statement}",
                "task_type": task,
                "trace_type": "recall_drill",
                "source": "course_prior_specification",
                "source_id": name,
                "verification_status": "accepted",
                "hidden_rule_used_in_completion": False,
                "token_length_proxy": len(statement.split()),
            })
    return items


def choose_sft_records(strategy: str, limit: int = MAX_TRAIN_RECORDS) -> list[dict[str, object]]:
    drills = build_recall_drill_records()
    full = FULL_TRACE_RECORDS
    if strategy in {"direct", "execution"}:
        return full[:limit]
    if strategy == "memorization":
        return drills[:limit]
    if strategy in {"joint", "two_stage"}:
        # Keep the Smoke budget fixed at 24 without duplicating records.
        full_budget = max(0, int(limit) - len(drills))
        return drills[:limit] + full[:full_budget]
    raise ValueError(f"unknown_sft_strategy:{strategy}")

def summarize_sft_data(strategy: str) -> dict[str, object]:
    records = choose_sft_records(strategy)
    by_type = Counter(str(row["trace_type"]) for row in records)
    return {
        "strategy": strategy,
        "records": len(records),
        "trace_type_counts": dict(by_type),
        "token_budget_proxy": int(sum(int(row["token_length_proxy"]) for row in records)),
        "task_counts": dict(Counter(str(row["task_type"]) for row in records)),
    }


SFT_DATA_SUMMARIES = [summarize_sft_data(name) for name in ("direct", "joint", "memorization", "execution", "two_stage")]
print(json.dumps(SFT_DATA_SUMMARIES, ensure_ascii=False, indent=2))

## 4. Chat Template、Label Masking 和 SFT Loss

训练样本由 User Prompt 和 Assistant Completion 构成。使用 Tokenizer 官方 Chat Template 生成两段序列，再用前缀长度定位 Completion；不手工拼接 Qwen 特殊 Token。`labels` 中 Prompt 和 Padding 均为 `-100`，因此 Cross Entropy 只看 Assistant Token。

In [ ]:
def encode_chat_example(tokenizer, record: dict[str, object], max_length: int = MAX_LENGTH) -> dict[str, list[int]]:
    messages = [{"role": "user", "content": str(record["prompt"])}]
    assistant = {"role": "assistant", "content": str(record["completion"])}
    try:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False, enable_thinking=True)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("Chat Template 未形成可定位的 Prompt 前缀，请检查 Tokenizer 配置")
    completion_ids = full_ids[len(prompt_ids):]
    if not completion_ids:
        raise ValueError("assistant completion 为空")
    input_ids = (prompt_ids + completion_ids)[:max_length]
    prompt_length = min(len(prompt_ids), len(input_ids))
    labels = [-100] * prompt_length + input_ids[prompt_length:]
    if not any(label != -100 for label in labels):
        raise ValueError("截断后没有可训练的 Completion Token")
    return {"input_ids": input_ids, "labels": labels, "prompt_length": prompt_length}


def collate_causal_examples(examples: list[dict[str, list[int]]], pad_token_id: int) -> dict[str, torch.Tensor]:
    max_len = max(len(example["input_ids"]) for example in examples)
    input_ids, labels, attention_mask = [], [], []
    for example in examples:
        pad = max_len - len(example["input_ids"])
        input_ids.append(example["input_ids"] + [pad_token_id] * pad)
        labels.append(example["labels"] + [-100] * pad)
        attention_mask.append([1] * len(example["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


def masked_sft_cross_entropy(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Next-token CE with Prompt/Padding labels already set to -100."""
    if logits.ndim != 3 or labels.ndim != 2:
        raise ValueError("logits must be [batch,time,vocab], labels must be [batch,time]")
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
    )


class ListSFTDataset(Dataset):
    def __init__(self, encoded_examples: list[dict[str, list[int]]]):
        self.examples = encoded_examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, index): return self.examples[index]


def run_sft_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: str, grad_accumulation_steps: int = 1) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses = []
    for step, batch in enumerate(loader):
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        logits = outputs.logits if hasattr(outputs, "logits") else outputs["logits"]
        loss = masked_sft_cross_entropy(logits, batch["labels"]) / grad_accumulation_steps
        loss.backward()
        if (step + 1) % grad_accumulation_steps == 0 or step + 1 == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        losses.append(float(loss.detach().cpu()) * grad_accumulation_steps)
    return float(np.mean(losses)) if losses else float("nan")

## 5. LoRA / QLoRA 配置与自定义训练入口

BF16 LoRA 和 NF4 QLoRA 只在 GPU 分支导入 PEFT、Transformers 和 bitsandbytes。Notebook 不安装依赖，也不在 CPU 阶段加载模型。QLoRA 只冻结 NF4 基座并训练 LoRA Adapter；激活、Adapter 梯度和优化器状态仍需显存，不能把量化理解成“所有东西都是 4-bit”。

In [ ]:
LORA_TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
LORA_CONFIG = {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05, "bias": "none", "target_modules": list(LORA_TARGET_MODULES)}


def build_peft_config(mode: str):
    if mode not in {"bf16_lora", "nf4_qlora"}:
        raise ValueError("mode must be bf16_lora or nf4_qlora")
    from peft import LoraConfig
    return LoraConfig(**LORA_CONFIG, task_type="CAUSAL_LM")


def build_quantization_config(mode: str):
    if mode != "nf4_qlora":
        return None
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )


def _model_load_kwargs(mode: str) -> dict[str, object]:
    if mode not in {"bf16_lora", "nf4_qlora"}:
        raise ValueError(mode)
    kwargs: dict[str, object] = {"trust_remote_code": True, "device_map": "auto"}
    if mode == "bf16_lora":
        kwargs["torch_dtype"] = torch.bfloat16
    else:
        kwargs["quantization_config"] = build_quantization_config(mode)
    return kwargs


def load_sft_model_and_tokenizer(mode: str = SMOKE_QUANTIZATION_MODE, adapter_path: Path | None = None):
    """Load the base model, then create or reload a PEFT Adapter.

    `adapter_path` is never passed to AutoModelForCausalLM; it is loaded with
    PeftModel.from_pretrained after the canonical Base Model has been loaded.
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, PeftModel, prepare_model_for_kbit_training
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)
    kwargs = _model_load_kwargs(mode)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, **kwargs)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if hasattr(model, "config"):
        model.config.use_cache = False
    if mode == "nf4_qlora":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    if adapter_path is None:
        model = get_peft_model(model, build_peft_config(mode))
    else:
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=True)
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
    return model, tokenizer


def _sha256_file(path: Path) -> str | None:
    import hashlib
    if not path.exists() or not path.is_file():
        return None
    return hashlib.sha256(path.read_bytes()).hexdigest()


def build_checkpoint_manifest(
    adapter_path: Path,
    mode: str,
    strategy: str,
    records: list[dict[str, object]],
    status: str,
    stage: str = "final",
) -> dict[str, object]:
    relative_adapter = str(adapter_path.relative_to(PROJECT_ROOT)) if adapter_path.is_relative_to(PROJECT_ROOT) else str(adapter_path)
    return {
        "schema_version": "nemotron-sft-checkpoint/v1",
        "status": status,
        "stage": stage,
        "base_model": {"id": MODEL_ID, "path": str(BASE_MODEL_PATH)},
        "tokenizer_path": str(TOKENIZER_PATH),
        "adapter_path": relative_adapter,
        "quantization": {
            "mode": mode,
            "load_in_4bit": mode == "nf4_qlora",
            "quant_type": "nf4" if mode == "nf4_qlora" else None,
            "double_quant": mode == "nf4_qlora",
            "compute_dtype": "bfloat16" if mode == "nf4_qlora" else "bfloat16",
        },
        "lora": {**LORA_CONFIG, "task_type": "CAUSAL_LM"},
        "training": {
            "strategy": strategy,
            "data_version": _sha256_file(TRAIN_PATH),
            "records": len(records),
            "token_budget_proxy": int(sum(int(row.get("token_length_proxy", 0)) for row in records)),
            "seed": SEED,
            "epochs": NUM_EPOCHS,
            "profile": TRAINING_PROFILE,
        },
    }


def write_checkpoint_manifest(manifest: dict[str, object], path: Path = SFT_MANIFEST_PATH) -> Path:
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def validate_checkpoint_manifest(manifest: dict[str, object]) -> None:
    required = {"schema_version", "status", "base_model", "tokenizer_path", "adapter_path", "quantization", "lora", "training"}
    missing = required - set(manifest)
    if missing:
        raise ValueError(f"checkpoint manifest missing: {sorted(missing)}")
    if manifest["schema_version"] != "nemotron-sft-checkpoint/v1":
        raise ValueError("unsupported checkpoint manifest schema")
    if manifest["base_model"].get("id") != MODEL_ID:
        raise ValueError("manifest base model does not match the fixed course model")
    if manifest["quantization"].get("mode") not in {"bf16_lora", "nf4_qlora"}:
        raise ValueError("manifest quantization mode is invalid")
    if int(manifest["lora"].get("r", 0)) <= 0:
        raise ValueError("manifest LoRA rank is invalid")


def _records_for_strategy(strategy: str) -> list[dict[str, object]]:
    return choose_sft_records(strategy, MAX_TRAIN_RECORDS)


def _train_records(model, tokenizer, records: list[dict[str, object]]) -> list[dict[str, float]]:
    encoded = [encode_chat_example(tokenizer, record) for record in records]
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    loader = DataLoader(ListSFTDataset(encoded), batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=lambda batch: collate_causal_examples(batch, pad_id))
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
    device = next(model.parameters()).device
    history = []
    for epoch in range(NUM_EPOCHS):
        history.append({"epoch": epoch + 1, "loss": run_sft_epoch(model, loader, optimizer, str(device), GRAD_ACCUMULATION_STEPS)})
    return history


def _save_adapter_checkpoint(model, tokenizer, output_dir: Path, manifest: dict[str, object]) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    write_checkpoint_manifest(manifest, SFT_MANIFEST_PATH)
    # Keep a copy next to the adapter for hand-off and audit.
    (output_dir / "checkpoint_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def run_sft_training(strategy: str = "direct", mode: str = SMOKE_QUANTIZATION_MODE) -> dict[str, object]:
    if not RUN_SFT:
        return {"status": "skipped_by_default", "reason": "RUN_SFT=False；DGX Spark 阶段不加载模型。"}
    if RUN_MODE == "smoke" and MAX_TRAIN_RECORDS != 24:
        raise ValueError("SFT Smoke 必须使用 24 条训练轨迹")
    started = time.perf_counter()
    stage_logs = []
    if strategy == "two_stage":
        stage1_records = build_recall_drill_records()
        stage1_dir = CHECKPOINT_ROOT / f"sft_two_stage_stage1_{mode}"
        model, tokenizer = load_sft_model_and_tokenizer(mode)
        stage1_history = _train_records(model, tokenizer, stage1_records)
        stage1_manifest = build_checkpoint_manifest(stage1_dir, mode, strategy, stage1_records, "completed", stage="stage1_recall_drill")
        _save_adapter_checkpoint(model, tokenizer, stage1_dir, stage1_manifest)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        stage2_records = FULL_TRACE_RECORDS[: max(0, MAX_TRAIN_RECORDS - len(stage1_records))]
        model, tokenizer = load_sft_model_and_tokenizer(mode, adapter_path=stage1_dir)
        stage2_history = _train_records(model, tokenizer, stage2_records)
        all_records = stage1_records + stage2_records
        output_dir = CHECKPOINT_ROOT / f"sft_two_stage_{mode}"
        history = stage1_history + stage2_history
        stage = "stage2_full_trace"
    else:
        all_records = _records_for_strategy(strategy)
        model, tokenizer = load_sft_model_and_tokenizer(mode)
        history = _train_records(model, tokenizer, all_records)
        output_dir = CHECKPOINT_ROOT / f"sft_{strategy}_{mode}"
        stage = "final"
    manifest = build_checkpoint_manifest(output_dir, mode, strategy, all_records, "completed", stage=stage)
    _save_adapter_checkpoint(model, tokenizer, output_dir, manifest)
    elapsed = time.perf_counter() - started
    return {"status": "completed", "strategy": strategy, "mode": mode, "records": len(all_records), "history": history, "training_time_seconds": elapsed, "adapter_path": str(output_dir), "manifest_path": str(SFT_MANIFEST_PATH), "peak_memory_allocated_mib": float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else None}


if RUN_SFT:
    sft_status = run_sft_training("two_stage", SMOKE_QUANTIZATION_MODE)
else:
    sft_status = {"status": "skipped_by_default", "reason": "当前阶段只做 CPU Smoke Test"}
print(json.dumps(sft_status, ensure_ascii=False, indent=2))

## 6. CPU Smoke Test

使用一个极小的 Embedding + Linear 语言模型验证：Prompt Label 全为 `-100`、Completion 有效监督、Padding 不参与 Loss、反向传播和优化器更新均正常。这个测试验证算法边界，不代表 Qwen3 的训练效果、显存或吞吐。

In [ ]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size: int = 32, hidden_size: int = 16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)
    def forward(self, input_ids, attention_mask=None):
        return type("Output", (), {"logits": self.lm_head(self.embedding(input_ids))})()


def toy_encoded(prompt_len: int, completion_len: int, vocab_size: int = 32) -> dict[str, list[int]]:
    input_ids = list(range(1, prompt_len + completion_len + 1))
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    return {"input_ids": input_ids, "labels": labels, "prompt_length": prompt_len}


torch.manual_seed(SEED)
toy_examples = [toy_encoded(3, 4), toy_encoded(2, 5)]
toy_batch = collate_causal_examples(toy_examples, pad_token_id=0)
assert all(value == -100 for value in toy_batch["labels"][0, :3].tolist())
assert all(value == -100 for value in toy_batch["labels"][1, :2].tolist())
assert (toy_batch["labels"] != -100).any()

toy_model = TinyCausalLM()
toy_optimizer = torch.optim.AdamW(toy_model.parameters(), lr=1e-2)
before = [parameter.detach().clone() for parameter in toy_model.parameters()]
loss = masked_sft_cross_entropy(toy_model(toy_batch["input_ids"]).logits, toy_batch["labels"])
assert torch.isfinite(loss)
loss.backward()
toy_optimizer.step()
after = [parameter.detach().clone() for parameter in toy_model.parameters()]
assert any(not torch.equal(left, right) for left, right in zip(before, after))

trace_audit_counts = Counter()
for record in FULL_TRACE_RECORDS[:20]:
    trace_audit_counts.update([record["trace_type"], "accepted"])

sft_cpu_report = {
    "status": "passed",
    "trace_records_checked": int(min(len(FULL_TRACE_RECORDS), 20)),
    "trace_audit": dict(trace_audit_counts),
    "label_masking": "prompt_and_padding_ignored",
    "loss_finite": True,
    "optimizer_update": True,
    "gpu_training": "not_run",
    "qlora": "configuration_only_not_run",
}
print(json.dumps(sft_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与后续实验

当前 CPU 测试只证明轨迹筛查、样本边界、Label Masking、Cross Entropy 和小模型更新正确。没有任何 GPU 训练结果，因此不能报告 Accuracy、Loss 曲线、显存或吞吐。

明天的 RTX 5090 顺序：先运行 Direct NF4 QLoRA Smoke，再运行 Joint / Two-stage；如显存允许，再执行 BF16 LoRA 对照。所有策略从同一数据版本出发，记录训练 Token Budget、训练时间、峰值显存、Adapter 保存/重载和 Validation 分任务指标。最佳 SFT Checkpoint 才能交给 DPO、PPO、GRPO。

## 7. Checkpoint Manifest 与 Adapter 交接 Smoke Test

CPU 阶段不加载 Qwen3；这里验证 SFT 交接所需的 Manifest 字段、固定 Base Model 和量化配置。GPU 阶段由 `run_sft_training` 保存 Adapter 后，再使用同一 Manifest 供 DPO/PPO/GRPO 读取。


In [ ]:
cpu_manifest = build_checkpoint_manifest(
    CHECKPOINT_ROOT / "cpu_fixture_adapter",
    "nf4_qlora",
    "two_stage",
    choose_sft_records("two_stage", MAX_TRAIN_RECORDS),
    "not_run",
)
validate_checkpoint_manifest(cpu_manifest)
assert cpu_manifest["base_model"]["id"] == MODEL_ID
assert cpu_manifest["quantization"]["load_in_4bit"] is True
assert cpu_manifest["lora"]["r"] == 8
manifest_test_path = RESULT_DIR / "cpu_checkpoint_manifest.json"
manifest_test_path.parent.mkdir(parents=True, exist_ok=True)
write_checkpoint_manifest(cpu_manifest, manifest_test_path)
assert json.loads(manifest_test_path.read_text())["schema_version"] == "nemotron-sft-checkpoint/v1"
sft_cpu_report = {"status":"passed", "smoke_records": len(choose_sft_records("two_stage", MAX_TRAIN_RECORDS)), "manifest": str(manifest_test_path.relative_to(PROJECT_ROOT)), "adapter_reload": "GPU_not_run", "qlora_prepare": "configuration_checked", "gpu_training": "not_run"}
(RESULT_DIR / "cpu_sft_smoke_report.json").write_text(json.dumps(sft_cpu_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(sft_cpu_report, ensure_ascii=False, indent=2))
